# NetSage — 01. Exploratory Data Analysis & Feature Investigation

**Dataset:** UCI RT-IoT2022 (Real-Time Internet of Things 2022)

### Objective:
1. Inspect IoT bidirectional flow characteristics.
2. Profile protocol distribution, packet rates, and payload volumes across traffic classes.
3. Analyze TCP flag distributions under SYN floods vs benign telemetry.
4. Assess multi-class imbalance and potential target leakage vectors.
5. Train baseline and tree-based classifiers to inspect feature attribution.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root in path
import sys
sys.path.append('..')

from src.data.loader import load_network_flow_csv
from src.data.validation import validate_and_profile_dataset
from src.ml.train import train_classification_model
from src.ml.explain import compute_global_feature_importance

data_path = Path('../data/sample_rt_iot2022.csv')
df, meta = load_network_flow_csv(data_path)
print(f"Loaded {meta['rows_count']} rows across {meta['columns_count']} columns.")
df.head()

## 2. Dataset Profiling & Validation

In [ ]:
profile, warnings = validate_and_profile_dataset(df, target_column_candidate='Attack_type')
print("Validation Warnings:", warnings)
print("Class Distribution:")
for cls, dist in profile['target_distribution'].items():
    print(f"  - {cls}: {dist['count']} ({dist['percentage']}%)")

## 3. Top Feature Correlations
Investigate collinearity between packet rates, byte rates, and durations.

In [ ]:
corr_pairs = pd.DataFrame(profile['top_correlations'])
corr_pairs.head(10)

## 4. Train Model and Inspect Global Feature Importance

In [ ]:
result = train_classification_model(df, model_type='random_forest', random_state=42)
print(f"Random Forest Holdout Accuracy: {result['metrics']['accuracy']:.4f}")
print(f"Macro F1-Score: {result['metrics']['macro_f1']:.4f}")

importance = compute_global_feature_importance(result['pipeline'], result['metadata'], top_n=10)
print("Top 10 Predictive Features:")
for item in importance['top_features']:
    print(f"  {item['feature']:<28}: {item['importance']:.4f}")

## 5. Analytical Observations & Limitations
- Features like `flow_SYN_flag_count`, `flow_pkts_per_sec`, and payload averages provide strong statistical separation between volumetric flooding attacks and low-throughput IoT telemetry.
- Note: Network classifiers trained purely on flow summaries can experience concept drift when new device protocols are introduced. Models should be monitored for distribution shifts over time.